In [10]:
import math, random, time, os
from argparse import Namespace
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

# Pick GPU if available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def get_config():
    return Namespace(
        seq_len=8,
        vocab_size=16,
        k_data=8,
        input_dim=64,
        hidden_dim=128,
        expansion=6,
        n_layers=2,
        dropout=0,
        batch_size=64,
        accum_steps=1,
        total_steps=10,
        eval_every=1000,
        lr=0.0005,
        seed=42,
        save_path="checkpoint.pt" # where models are saved
    )

CFG = get_config()

Device: cpu


In [11]:
class SelectiveCopyDataset(Dataset):
    def __init__(self, cfg, n_examples=10000):
        self.cfg = cfg
        self.n_examples = n_examples
        self.seq_len = cfg.seq_len
        self.vocab_size = cfg.vocab_size
        self.k_data = cfg.k_data
        self.total_len = self.seq_len + self.k_data
        self.pad_id = -100
        self.decoder_query_id = cfg.vocab_size

    def __len__(self):
        return self.n_examples # number of examples in the dataset

    def __getitem__(self, idx):
        # randomly select k_data positions in the input sequence to later predict
        pos = np.random.choice(self.seq_len, self.k_data, replace=False)
        pos.sort()
        data_tokens = np.random.randint(0, self.vocab_size, size=(self.k_data,))
        inp = np.random.randint(0, self.vocab_size, size=(self.seq_len,))
        inp[pos] = data_tokens
        inp_full = np.concatenate([inp, np.full(self.k_data, self.decoder_query_id)])
        target = np.full(self.total_len, self.pad_id, dtype=np.int64)
        target[self.seq_len:] = data_tokens
        return torch.from_numpy(inp_full).long(), torch.from_numpy(target).long()

_LOG_CLAMP_MIN = -50.0 # avoid overflow in torch.exp()
def safe_exp_log(log_w):
    return torch.exp(torch.clamp(log_w, min=_LOG_CLAMP_MIN))

def combine_logvec_pairs(loga_prev, v_prev, loga_curr, v_curr):
    loga_out = loga_curr + loga_prev
    a_curr = safe_exp_log(loga_curr)
    v_out = a_curr * v_prev + v_curr
    return loga_out, v_out

def parallel_prefix_logvec(loga, v):
    T, B, H = loga.shape
    L = loga.clone()
    V = v.clone()
    step = 1
    while step < T:
        src = torch.arange(step, T, device=loga.device)
        L_prev = L[src - step]
        V_prev = V[src - step]
        L_curr = L[src]
        V_curr = V[src]
        L_comb, V_comb = combine_logvec_pairs(L_prev, V_prev, L_curr, V_curr)
        L[src] = L_comb
        V[src] = V_comb
        step *= 2
    return V

In [12]:
class MinGRUBlockMatrix(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.z = nn.Linear(input_dim, hidden_dim)
        self.h_lin = nn.Linear(input_dim, hidden_dim)

    # standard sequential GRU forward
    def forward_sequential(self, x):
        T,B,_ = x.shape
        h = torch.zeros(B, self.h_lin.out_features, device=x.device)
        out = []
        for t in range(T):
            z = torch.sigmoid(self.z(x[t]))
            tilde = self.h_lin(x[t])
            h = (1 - z) * h + z * tilde
            out.append(h)
        return torch.stack(out)

    # parallel GRU forward
    def forward_parallel(self, x):
        T,B,_ = x.shape
        z = torch.sigmoid(self.z(x))
        tilde = self.h_lin(x)
        alpha = torch.clamp(1.0 - z, min=1e-12)
        loga = torch.log(alpha)
        v = z * tilde
        v_prod = parallel_prefix_logvec(loga, v)
        return v_prod


class MinLSTMBlockMatrix(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.f = nn.Linear(input_dim, hidden_dim)
        self.i = nn.Linear(input_dim, hidden_dim)
        self.tilde = nn.Linear(input_dim, hidden_dim)

    # standard sequential LSTM forward
    def forward_sequential(self, x):
        T,B,_ = x.shape
        h = torch.zeros(B, self.tilde.out_features, device=x.device)
        out = []
        for t in range(T):
            f = torch.sigmoid(self.f(x[t]))
            i = torch.sigmoid(self.i(x[t]))

            # Normalize gates to prevent hidden state explosion
            denom = f + i + 1e-12
            f_p = f / denom
            i_p = i / denom

            tilde = self.tilde(x[t])
            h = f_p * h + i_p * tilde
            out.append(h)
        return torch.stack(out)

    # parallel LSTM forward
    def forward_parallel(self, x):
        T,B,_ = x.shape
        f = torch.sigmoid(self.f(x))
        i = torch.sigmoid(self.i(x))
        denom = f + i + 1e-12
        f_p = f / denom
        i_p = i / denom
        tilde = self.tilde(x)
        f_p_clamped = torch.clamp(f_p, min=1e-12)
        loga = torch.log(f_p_clamped)
        v = i_p * tilde
        v_prod = parallel_prefix_logvec(loga, v)
        return v_prod

class MinRNNLayer(nn.Module):
    def __init__(self, input_dim, hidden_dim, expansion, cell, dropout=0.1):
        super().__init__()
        self.cell_type = cell
        exp_dim = hidden_dim * expansion
        self.expand = nn.Linear(input_dim, exp_dim)
        if cell == "gru":
            self.cell = MinGRUBlockMatrix(exp_dim, exp_dim)
        else:
            self.cell = MinLSTMBlockMatrix(exp_dim, exp_dim)
        self.down = nn.Linear(exp_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(hidden_dim)

    def forward_sequential(self, x):
        T,B,_ = x.shape
        xe = self.expand(x.view(T*B, -1)).view(T,B,-1)
        h = self.cell.forward_sequential(xe)
        h = self.down(h)
        h = self.dropout(h)
        out = self.norm(x + h)
        return out

    def forward_parallel(self, x):
        T,B,_ = x.shape
        xe = self.expand(x.view(T*B, -1)).view(T,B,-1)
        h = self.cell.forward_parallel(xe)
        h = self.down(h)
        h = self.dropout(h)
        out = self.norm(x + h)
        return out
    
class MinRNNLayerWithLinearAttention(nn.Module):
    def __init__(self, input_dim, hidden_dim, expansion, cell, dropout=0.1):
        super().__init__()
        self.cell_type = cell
        exp_dim = hidden_dim * expansion

        self.expand = nn.Linear(input_dim, exp_dim)
        if cell == "gru":
            self.cell = MinGRUBlockMatrix(exp_dim, exp_dim)
        else:
            self.cell = MinLSTMBlockMatrix(exp_dim, exp_dim)
        self.down = nn.Linear(exp_dim, hidden_dim)

        self.q_proj = nn.Linear(hidden_dim, hidden_dim)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim)

        self.feature_map = lambda x: F.elu(x) + 1

        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(hidden_dim)

    def minrnn_forward(self, x):
        T, B, _ = x.shape
        xe = self.expand(x.reshape(T * B, -1)).view(T, B, -1)
        h = self.cell.forward_parallel(xe)
        h = self.down(h)
        return self.dropout(h)

    def linear_attention(self, h):
        """
        h: (T, B, H)
        Returns a_t for each t with linear attention:
        a_t = (φ(q_t)·S_t) / (φ(q_t)·Z_t)
        """
        T, B, H = h.shape

        Q = self.feature_map(self.q_proj(h))
        K = self.feature_map(self.k_proj(h))
        V = self.v_proj(h)

        # S_t = sum_{i<=t} K_i * V_i
        S = torch.cumsum(K * V, dim=0)

        # Z_t = sum_{i<=t} K_i
        Z = torch.cumsum(K, dim=0)

        # Numerator = Q_t dot S_t
        num = (Q * S).sum(dim=-1, keepdim=True)

        # Denominator = Q_t dot Z_t
        denom = (Q * Z).sum(dim=-1, keepdim=True) + 1e-8

        A = num / denom

        return A * V

    def forward_parallel(self, x):
        h = self.minrnn_forward(x)
        a = self.linear_attention(h)
        out = self.norm(x + h + a)

        return out

    def forward_sequential(self, x):
        return self.forward_parallel(x)

In [13]:
class StackedRNN(nn.Module):
    def __init__(self, cfg, cell_type):
        super().__init__()
        self.cfg = cfg
        D = cfg.input_dim
        H = cfg.hidden_dim
        self.token_embed = nn.Embedding(cfg.vocab_size + 1, D)
        if D != H:
            self.in_proj = nn.Linear(D, H)
        else:
            self.in_proj = nn.Identity()
        
        if cell_type.lower() == "gru":
            self.rnn = nn.GRU(input_size=H, hidden_size=H, num_layers=cfg.n_layers)
        else:
            self.rnn = nn.LSTM(input_size=H, hidden_size=H, num_layers=cfg.n_layers)
        
        self.readout = nn.Linear(H, cfg.vocab_size)

    # since run() only calls forward_parallel, we have forward_parallel call forward
    def forward_parallel(self, tokens):
        return self.forward(tokens)
    
    def forward(self, tokens):
        T, B = tokens.shape
        x = self.token_embed(tokens)
        x = self.in_proj(x)
        out, _ = self.rnn(x)
        logits = self.readout(out)
        return logits

class StackedMinRNN(nn.Module):
    def __init__(self, cfg, cell, attention=False):
        super().__init__()
        self.cfg = cfg
        D = cfg.input_dim
        H = cfg.hidden_dim
        self.token_embed = nn.Embedding(cfg.vocab_size + 1, D)
        if D != H:
            self.in_proj = nn.Linear(D, H)
        else:
            self.in_proj = nn.Identity()
        
        if attention:
            self.layers = nn.ModuleList([
                MinRNNLayerWithLinearAttention(H, H, cfg.expansion, cell=cell, dropout=cfg.dropout)
                for _ in range(cfg.n_layers)
            ])
        else:
            self.layers = nn.ModuleList([
                MinRNNLayer(H, H, cfg.expansion, cell=cell, dropout=cfg.dropout)
                for _ in range(cfg.n_layers)
            ])

        self.readout = nn.Linear(H, cfg.vocab_size)

    def forward_sequential(self, tokens):
        T,B = tokens.shape
        x = self.token_embed(tokens)
        x = self.in_proj(x)
        for layer in self.layers:
            x = layer.forward_sequential(x)
        logits = self.readout(x)
        return logits

    def forward_parallel(self, tokens):
        T,B = tokens.shape
        x = self.token_embed(tokens)
        x = self.in_proj(x)
        for layer in self.layers:
            x = layer.forward_parallel(x)
        logits = self.readout(x)
        return logits

In [ ]:
def compute_accuracy(pred_logits, target, ignore_index=-100):
    preds = pred_logits.argmax(dim=-1)
    mask = (target != ignore_index)
    if mask.sum() == 0:
        return 0.0
    
    # Count correct predictions only where mask == True
    correct = (preds == target) & mask
    return (correct.sum().float() / mask.sum().float()).item()

def train_one_epoch(model, dataloader, opt, cfg, step_state):
    model.train()
    total_loss = 0.0
    total_acc = 0.0
    total_examples = 0
    grad_accum = cfg.accum_steps
    it = 0
    for tokens, targets in dataloader:
        tokens = tokens.transpose(0,1).to(device)
        targets = targets.transpose(0,1).to(device)
        logits = model.forward_parallel(tokens)
        loss = F.cross_entropy(
            logits.reshape(-1, cfg.vocab_size),
            targets.reshape(-1),
            ignore_index=-100
        )

        loss = loss / grad_accum
        loss.backward()
        if (it+1) % grad_accum == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            opt.zero_grad()

        # Track total loss (undo grad_accum division)
        total_loss += loss.item() * grad_accum

        # Accuracy for this batch (multiply by batch size to weight properly)
        total_acc += compute_accuracy(logits, targets, ignore_index=-100) * tokens.shape[1]
        total_examples += tokens.shape[1]
        it += 1
        step_state['step'] += 1
        if step_state['step'] >= cfg.total_steps:
            break
    return total_loss / max(1, it), total_acc / max(1, total_examples)

def evaluate(model, dataloader, cfg):
    model.eval()
    tot_acc = 0.0
    tot = 0
    with torch.no_grad():
        for tokens, targets in dataloader:
            tokens = tokens.transpose(0,1).to(device)
            targets = targets.transpose(0,1).to(device)
            logits = model.forward_parallel(tokens)
            acc = compute_accuracy(logits, targets, ignore_index=-100)
            tot_acc += acc * tokens.shape[1]
            tot += tokens.shape[1]
    return tot_acc / tot if tot>0 else 0.0

In [ ]:
def run(cfg, model_class, model_type, attention=False):
    # Set random seeds for reproducibility
    torch.manual_seed(cfg.seed)
    np.random.seed(cfg.seed)
    random.seed(cfg.seed)

    train_ds = SelectiveCopyDataset(cfg, n_examples=200000)
    val_ds = SelectiveCopyDataset(cfg, n_examples=2000)

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False)

    if model_class == 'minrnn':        
        model = StackedMinRNN(cfg, cell=model_type, attention=attention).to(device)
    else:
        model = StackedRNN(cfg, cell_type=model_type).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr)

    # training loop
    step_state = {'step': 0}
    best_val = 0.0 # For saving best-performing model
    start_time = time.time()
    while step_state['step'] < cfg.total_steps:
        train_loss, train_acc = train_one_epoch(model, train_loader, opt, cfg, step_state)
        if step_state['step'] % cfg.eval_every == 0 or step_state['step'] >= cfg.total_steps:
            val_acc = evaluate(model, val_loader, cfg)
            print(f"[step {step_state['step']}/{cfg.total_steps}] train_loss={train_loss:.4f} train_acc={train_acc:.4f} val_acc={val_acc:.4f}")
            if val_acc > best_val:
                best_val = val_acc
                torch.save(model.state_dict(), cfg.save_path)

    total_time = time.time() - start_time
    print(f"Training done. Best val acc: {best_val:.4f}. Time elapsed: {total_time:.1f} secs, or {total_time/60:.2f} min.")

if __name__ == "__main__":
    # configure model type and class here
    model_type = "gru"
    model_class = "minrnn"
    atn = True

    print("Config:\n", CFG)
    print("Model type:", model_type)
    print("Attention:", atn)
    
    run(CFG, model_class, model_type, atn)


Config:
 Namespace(seq_len=8, vocab_size=16, k_data=8, input_dim=64, hidden_dim=128, expansion=6, n_layers=2, dropout=0, batch_size=64, accum_steps=1, total_steps=10, eval_every=1000, lr=0.0005, seed=42, save_path='checkpoint.pt')
Model type: gru
Attention: True
[step 10/10] train_loss=2.8392 train_acc=0.0619 val_acc=0.0617
Training done. Best val acc: 0.0617. Time elapsed: 3.2 secs, or 0.05 min.
